## Tool definition

In [4]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain.tools import tool

@tool
def usd_to_euro(x: float) -> float:
    """Calculate EUR equivalent after exchanging USD to EUR"""
    return x * 0.85

In [6]:
@tool("usd_to_euro")
def tool1(x: float) -> float:
    """Calculate EUR equivalent after exchanging USD to EUR"""
    return x * 0.85

In [ ]:
@tool("usd_to_euro", description="Calculate EUR equivalent after exchanging USD to EUR")
def tool1(x: float) -> float:
    return x * 0.85

In [7]:
tool1.invoke({"x": 467})

396.95

## Adding to agents

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5.4-nano",
    tools=[tool1],
    system_prompt="You are a currency exchange wizard. Use your tools to calculate the EURO equivalent of the USD amount."
)

In [9]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How much is $20.212 in euro?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

$20.212 is **€17.1802**.


In [10]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='How much is $20.212 in euro?', additional_kwargs={}, response_metadata={}, id='cd78f921-53a6-487a-87d8-c5708a38d748'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 162, 'total_tokens': 184, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DX87CyTjAeSBKvl7aETVaBB2tpySf', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db0d7-3bf1-7791-aa84-5bf7c471e0eb-0', tool_calls=[{'name': 'usd_to_euro', 'args': {'x': 20.212}, 'id': 'call_20gHRuSPBLkRcPE5lLt1WVhv', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 162, 'output_tokens': 22, 'total_token

In [11]:
print(response["messages"][1].tool_calls)

[{'name': 'usd_to_euro', 'args': {'x': 20.212}, 'id': 'call_20gHRuSPBLkRcPE5lLt1WVhv', 'type': 'tool_call'}]


## Without web search

In [14]:
from dotenv import load_dotenv

load_dotenv()

True

In [16]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5.4-nano"
)

In [17]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Who performed at the halftime show of superbowl 2026?")

response = agent.invoke(
    {"messages": [question]}
)

In [18]:
print(response['messages'][-1].content)

Super Bowl 2026 hasn’t happened yet (so the halftime performer isn’t known as of now).  

If you tell me which “Super Bowl 2026” you mean (the game number/date), I can check the latest announced performer once it’s available.


In [19]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

In [20]:
print(response['messages'][-1].content)

My training knowledge is based on data available up to **August 2025**.  

That means anything after that date may not be included in my built-in knowledge. However, I can still help with things like:
- interpreting newer information you provide,
- reasoning from first principles,
- summarizing or explaining content you paste in,
- and helping you find up-to-date answers if you have sources/links.


## Add web search tool

In [21]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

In [22]:
web_search.invoke("Who performed at the halftime show of superbowl 2026?")

{'query': 'Who performed at the halftime show of superbowl 2026?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.cbsnews.com/news/who-is-performing-super-bowl-halftime-show-2026/',
   'title': "Who performed at the Super Bowl halftime show in 2026? Here's a ...",
   'content': 'On the heels of his historic night at the Grammy Awards, Bad Bunny took the stage for the halftime show at the 2026 Super Bowl on Sunday, as the New England Patriots and the Seattle Seahawks meet in a rematch of Super Bowl XLIX. The NFL, Apple Music and Roc Nation announced in September that Bad Bunny would be this year\'s halftime show headliner. All of that," Bad Bunny said at a Thursday news conference hosted by Apple Music, which is sponsoring the halftime show. * Read more: Will Bad Bunny be paid for his Super Bowl halftime performance? Bad Bunny was the first native Spanish speaker to headline the Super Bowl halftime show, and his selection prompted critici

In [23]:
agent = create_agent(
    model="gpt-5.4-nano",
    tools=[web_search]
)

question = HumanMessage(content="Who performed at the halftime show of superbowl 2026?")

response = agent.invoke(
    {"messages": [question]}
)

In [24]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who performed at the halftime show of superbowl 2026?', additional_kwargs={}, response_metadata={}, id='1043429e-fb9a-4c10-9fce-3ae982816856'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 137, 'total_tokens': 161, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DX8DpN4pVyjQOWpXepTtiBBjhAosA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db0dd-85ba-7830-8820-ebe77b179889-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'Super Bowl 2026 halftime show performers'}, 'id': 'call_2WsavjTMCZmP5Eu9RrP3OXFX', 'type': 'tool_call'}], invalid_tool_calls=[], usage_

In [25]:
pprint(response['messages'][-1].content)

('The halftime show at **Super Bowl 2026 (Super Bowl LX)** was headlined by '
 '**Bad Bunny**.')
